In [0]:
%sql
CREATE TABLE my_files.my_schema.change_feed_data(first_name string,age long) using delta TBLPROPERTIES(delta.enableChangeDataFeed=true)

In [0]:

df=spark.createDataFrame([("bob",23),("sue",98),("jim",27)]).toDF("first_name","age")
df.write.mode("append").format("delta").saveAsTable("my_files.my_schema.change_feed_data")

In [0]:
%sql
select * from my_files.my_schema.change_feed_data

first_name,age
bob,23
sue,98
jim,27


In [0]:
%sql
delete from my_files.my_schema.change_feed_data where first_name='bob'

num_affected_rows
1


In [0]:
%sql
update my_files.my_schema.change_feed_data set age=35 where first_name='sue'

num_affected_rows
1


In [0]:
df=spark.read.format("delta")\
    .option("readChangeFeed","true")\
    .option("startingVersion",0)\
    .table('my_files.my_schema.change_feed_data')
display(df)

first_name,age,_change_type,_commit_version,_commit_timestamp
sue,98,update_preimage,4,2026-03-10T06:35:46.000Z
sue,35,update_postimage,4,2026-03-10T06:35:46.000Z
bob,23,insert,1,2026-03-10T06:29:57.000Z
sue,98,insert,1,2026-03-10T06:29:57.000Z
jim,27,insert,1,2026-03-10T06:29:57.000Z
bob,23,delete,2,2026-03-10T06:34:42.000Z
